# Granite Speech Demo — full stack in Colab (single audio model)

Spin up a real-time voice assistant powered by IBM Granite 4.1 — entirely inside a Colab notebook. One cell brings up a **single** vLLM server (an audio-enabled Granite Switch checkpoint), the Pipecat backend, and the Next.js frontend, then prints a public URL you open in your browser to start talking.

**Browser mic → WebRTC → Granite Switch (audio model) → Kokoro TTS → browser speaker.**

This is the **single-model** variant of the [granite-speech-demo](https://github.com/generative-computing/mellea-demos/tree/main/2026-granite-speech) reference implementation (backend on the `asr` branch).

## What this demo is

One WebRTC conversation served by **one model**. The audio-enabled **Granite Switch** checkpoint takes the user's speech directly: it transcribes the audio internally (a small ASR model embedded in the same vLLM process) and generates the spoken answer in a single request — no separate speech-to-text server, no extra orchestration hop. The response streams token-by-token into Kokoro TTS so the assistant starts speaking before the full answer is generated.

> This variant trades the previous two-model setup (separate Granite Speech STT + a Mellea-orchestrated Best-of-N/validation LLM) for the simplest possible path: **audio in → one model → answer out.** It showcases that Granite Switch can accept audio as a single deployable model.

## Prerequisites

- **GPU runtime with ~16+ GiB free** (e.g. Colab A100/L4). The audio model = the LLM weights + a small embedded Whisper.
- **A composed, audio-enabled checkpoint.** Our audio model isn't on the Hub — compose it once with `--enable-audio` and point `MODEL_PATH` at it (see the configuration cell).
- **HuggingFace read token.** Free; create one at https://huggingface.co/settings/tokens. Add it as a Colab Secret named `HF_TOKEN`. Used for downloading model weights *and* minting per-session WebRTC TURN credentials.
- **Browser:** Chrome, Edge, or Firefox. Safari may behave oddly with WebRTC.

## How long this takes

- **First run on a fresh runtime: ~6–8 min** (model download/compose dominates).
- **Subsequent runs with weights cached: ~3 min.**

## What to do

1. Set the `HF_TOKEN` Colab Secret.
2. Switch the runtime to a GPU (Runtime → Change runtime type).
3. Make sure `MODEL_PATH` (configuration cell) points at your composed audio checkpoint.
4. **Runtime → Run all.**
5. When the last cell prints a `*.trycloudflare.com` URL, open it, allow mic access, and start talking.

If anything goes wrong, scroll to the bottom — there's a troubleshooting section and a kill-switch cell.

## 1 · Install dependencies (~3 min)

Clones the repo, installs Python deps via `uv`, installs frontend deps via `npm`, and downloads the `cloudflared` binary used for the public tunnel.

In [ ]:
import subprocess, os, shutil

def sh(cmd, **kwargs):
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, **kwargs)

# Re-runnable: nuke any stale clones so we don't trip on existing dirs.
shutil.rmtree("mellea-demos", ignore_errors=True)
shutil.rmtree("/tmp/granite-switch", ignore_errors=True)

# Colab's default Ubuntu repo has Node 12, which is too old for Next.js
# (chokes on optional-chaining). Install Node 20 from NodeSource instead.
sh("curl -fsSL https://deb.nodesource.com/setup_20.x | bash -")
sh("apt-get -qq install -y nodejs")

# The `asr` branch (on this fork) carries the single-model voice pipeline
# (audio -> answer in one request; no separate STT, no Mellea stage).
sh("git clone -b asr https://github.com/aviv1ron1/mellea-demos")
os.chdir("mellea-demos/2026-granite-speech")
print("cwd:", os.getcwd())

sh("pip install -q uv")
sh("uv sync")

# IMPORTANT: uv sync creates .venv, but `uv pip install` by default targets
# the system Python. Pin every subsequent install to the project venv.
VENV_PY = os.path.abspath(".venv/bin/python")
assert os.path.exists(VENV_PY), f"venv missing: {VENV_PY}"

# 1. mellea 0.6.0 — still installed (the package imports it elsewhere), but the
#    single-model voice path no longer uses the Mellea LLM stage.
sh(f"uv pip install --python {VENV_PY} 'mellea[all]==0.6.0'")

# 2. vllm + the right transformers floor + granite_switch model registration
#    (the audio-enabled GraniteSwitch architecture + its ASR processor live here).
sh("git clone -b asr-switch https://github.com/generative-computing/granite-switch /tmp/granite-switch")
assert os.path.exists("/tmp/granite-switch/pyproject.toml"), "granite-switch clone failed"
sh(f"uv pip install --python {VENV_PY} -e '/tmp/granite-switch[vllm,audio]'")

# 3. vllm audio deps (the ASR cascade needs librosa + soundfile to decode audio).
sh(f"uv pip install --python {VENV_PY} librosa soundfile")

# 4. Final transformers pin (5.5.1 is what we tested against Granite Switch).
sh(f"uv pip install --python {VENV_PY} 'transformers==5.5.1'")

sh("cd frontend && npm install --silent")
sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
sh("chmod +x /usr/local/bin/cloudflared")

# Sanity checks — explicitly use the venv's Python so we're checking the right env.
sh(f"{VENV_PY} -c 'import vllm; print(\"vllm version:\", vllm.__version__)'")
sh(f"{VENV_PY} -c 'import granite_switch.hf'")
sh(f"{VENV_PY} -c 'import transformers; v = transformers.__version__; assert v == \"5.5.1\", \"got \" + v + \", wanted 5.5.1\"; print(\"transformers OK:\", v)'")
sh(f"{VENV_PY} -c 'import librosa, soundfile; print(\"vllm audio deps OK (librosa\", librosa.__version__, \"/ soundfile\", soundfile.__version__, \")\")'")

print("\n✅ Install complete")

## 2 · Configure secrets (instant)

Reads `HF_TOKEN` from Colab Secrets and exports it. Used for both HuggingFace model downloads and per-session TURN credential minting (see [TURN setup](https://turn.fastrtc.org/) — Cloudflare-backed, 10GB/mo free per HF token).

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("✅ HF_TOKEN configured — TURN credentials will be minted per-session")

## 3 · Configure the assistant (optional)

The backend reads two env vars to customize what the assistant knows and how it behaves:

- **`PROMPT_FILE`** — path to a `.txt` file with the system prompt. Defaults to [`prompts/granite.txt`](https://github.com/generative-computing/mellea-demos/blob/main/2026-granite-speech/prompts/granite.txt), which casts the assistant as Granite, IBM's real-time speech assistant.
- **`DOCUMENTS_DIR`** — path to a directory of `.txt` files. Each file becomes a grounding document the LLM can cite. The repo ships with [`docs/`](https://github.com/generative-computing/mellea-demos/tree/main/2026-granite-speech/docs) (Granite model cards, Mellea overview, demo architecture).

Paths are resolved relative to the project root (`mellea-demos/2026-granite-speech/`).

**To use your own:** edit the cell below before running it. Drop your prompt file and/or doc directory anywhere reachable from the runtime — e.g. upload via the Colab file browser, or `!wget` from a URL — then point the env vars at them.

In [ ]:
import os

# Path to the COMPOSED, audio-enabled Granite Switch checkpoint (the single
# model this demo serves). Built by patching granite-switch-4.1-3b-preview with
# the <|audio|> token + audio chat-template + asr_enabled config, and saved to
# COS — available on Vela at the mount below.
#   (On Colab instead, point this at a local path you composed/uploaded.)
os.environ["MODEL_PATH"] = "/danieloh_cos/avivron/granite-switch/granite-switch-audio"

# System prompt for the assistant (sent with each audio request).
os.environ["PROMPT_FILE"] = "prompts/granite.txt"

print(f"MODEL_PATH  = {os.environ['MODEL_PATH']}")
print(f"PROMPT_FILE = {os.environ['PROMPT_FILE']}")

## 4 · Launch the vLLM model server (~2-4 min cold, ~30s cached)

**One** vLLM process — your composed, audio-enabled **Granite Switch** checkpoint (`MODEL_PATH`) on **port 8000**. It transcribes the incoming audio *internally* (Whisper, on the `asr_device` baked into the checkpoint) and generates the answer in the same request, so there's no separate STT server.

Runs in the background; logs stream to `logs/vllm-audio.log`. The cell blocks until the server responds on `/v1/models`.

In [ ]:
import os
import subprocess
import time
import urllib.request
import urllib.error

os.makedirs("logs", exist_ok=True)

VENV_VLLM = os.path.abspath(".venv/bin/vllm")
assert os.path.exists(VENV_VLLM), f"vllm not installed in venv: {VENV_VLLM}"

MODEL_PATH = os.environ.get("MODEL_PATH", "/content/granite-switch-audio")
SERVED_NAME = "granite-switch-audio"
assert os.path.exists(os.path.join(MODEL_PATH, "config.json")), (
    f"No composed audio checkpoint at {MODEL_PATH}. Compose one with --enable-audio "
    "first (see the configuration cell above)."
)

# Pre-flight: kill any stale vllm processes, verify enough free GPU memory.
subprocess.run("pkill -9 -f vllm || true", shell=True)
time.sleep(3)
free_mem = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"]
).decode().strip().splitlines()[0]
free_gib = int(free_mem) / 1024
print(f"GPU free memory: {free_gib:.1f} GiB")
if free_gib < 16:
    raise RuntimeError(
        f"Only {free_gib:.1f} GiB free on the GPU — need >=16 for the audio model "
        "(LLM + Whisper). Free the GPU (kill-switch cell) and retry."
    )

def _tail(path: str, n: int = 80) -> str:
    try:
        with open(path) as f:
            return "".join(f.readlines()[-n:])
    except FileNotFoundError:
        return "(log file missing)"

def wait_for(url: str, name: str, proc: subprocess.Popen, log_path: str, timeout: int = 1200) -> None:
    """Poll until the URL responds. Bails out early if the process dies."""
    start = time.time()
    last_err = None
    while time.time() - start < timeout:
        rc = proc.poll()
        if rc is not None:
            raise RuntimeError(
                f"{name} exited early with code {rc}. Last log lines:\n"
                + "-" * 60 + "\n" + _tail(log_path) + "-" * 60
            )
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if 200 <= r.status < 300:
                    print(f"✅ {name} ready ({int(time.time() - start)}s)")
                    return
        except urllib.error.HTTPError as e:
            print(f"✅ {name} ready ({int(time.time() - start)}s, status {e.code})")
            return
        except (urllib.error.URLError, ConnectionError, TimeoutError) as e:
            last_err = e
        time.sleep(5)
    raise TimeoutError(
        f"{name} did not become ready in {timeout}s. Last error: {last_err}.\n"
        f"Last log lines:\n" + "-" * 60 + "\n" + _tail(log_path) + "-" * 60
    )

# ONE server now: the audio-enabled Granite Switch model. It transcribes the
# incoming audio internally (Whisper on the asr_device baked into the checkpoint
# config) and generates the answer — so there's no separate STT server.
audio_log = open("logs/vllm-audio.log", "w")
print("⏳ Starting Granite Switch (audio) vLLM (loads weights + Whisper, ~2-4 min)...")
switch_proc = subprocess.Popen(
    [
        VENV_VLLM, "serve", MODEL_PATH,
        "--served-model-name", SERVED_NAME,
        # One model now, so it can use more of the GPU than the old 0.4 split.
        "--gpu-memory-utilization", "0.6",
        "--max-model-len", "8192",
        "--port", "8000",
    ],
    stdout=audio_log, stderr=subprocess.STDOUT,
)
wait_for("http://127.0.0.1:8000/v1/models", "Granite Switch (audio)", switch_proc, "logs/vllm-audio.log", timeout=1200)

print("✅ Audio model server is up (one model — STT is built in)")

## 5 · Launch backend + frontend (~30s)

- **Pipecat backend** on port 7860 (FastAPI + SmallWebRTC signaling).
- **Next.js frontend** on port 3000 (proxies WebRTC signaling to the backend in-process).

The backend reads `HF_TOKEN` and uses it to mint a TURN relay credential per session — that's how WebRTC media reaches your browser through the cloudflared tunnel.

In [ ]:
import os
import subprocess
import time
import urllib.request
import urllib.error

VENV_PY = os.path.abspath(".venv/bin/python")

# Build the frontend in production mode. Dev mode (`npm run dev`) tries to
# open a webpack-hmr WebSocket back through the cloudflared tunnel, which
# tunnels poorly and triggers dynamic-import failures that leave the chat
# UI blank. Prod mode is a static-served bundle — no HMR, no SSR weirdness.
print("⏳ Building frontend (prod mode, ~30-60s)...")
subprocess.run(
    "cd frontend && rm -rf .next && npm run build 2>&1 | tail -10",
    shell=True, check=True,
)

backend_env = {**os.environ}
backend_env.setdefault("HOST", "127.0.0.1")
backend_env.setdefault("PORT", "7860")
# Point the backend's single audio service at our one model server. The
# AudioLLMService sends the user's audio (input_audio) straight here and streams
# the answer — no separate STT endpoint.
backend_env["LLM_URL"] = "http://127.0.0.1:8000/v1"
backend_env["LLM_MODEL"] = "granite-switch-audio"
# PROMPT_FILE is set in the configuration cell above and inherited via os.environ.

backend_log = open("logs/backend.log", "w")
backend_proc = subprocess.Popen(
    [VENV_PY, "-m", "granite_speech_demo.server"],
    env=backend_env,
    stdout=backend_log, stderr=subprocess.STDOUT,
)

frontend_env = {**os.environ, "PIPECAT_BACKEND_URL": "http://127.0.0.1:7860"}
frontend_log = open("logs/frontend.log", "w")
frontend_proc = subprocess.Popen(
    ["npm", "run", "start"],
    cwd="frontend",
    env=frontend_env,
    stdout=frontend_log, stderr=subprocess.STDOUT,
)

wait_for("http://127.0.0.1:7860/api/ivr/config", "Pipecat backend", backend_proc, "logs/backend.log", timeout=120)
wait_for("http://127.0.0.1:3000", "Next.js frontend", frontend_proc, "logs/frontend.log", timeout=120)
print("✅ Backend + frontend are up")

## 6 · Open the public URL and talk

Starts a Cloudflare Quick Tunnel to expose `localhost:3000` on a public `*.trycloudflare.com` URL. The tunnel handles WebRTC *signaling* (HTTP/WebSocket); the *media* path goes through the TURN relay minted by the backend, so audio works even though the Colab runtime has no public IP.

**One tunnel is enough** — the frontend talks to the backend in-process via Next.js API routes.

**Heads up:** the first interaction will feel slow. There's one-time setup that runs when the environment and networking first spin up (TURN credentials, WebRTC negotiation, model warmup). Subsequent turns are much faster.

In [ ]:
import re
import subprocess
import time

tunnel_log_path = "logs/cloudflared.log"
tunnel_log = open(tunnel_log_path, "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:3000", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

url_re = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
public_url = None
deadline = time.time() + 60
while time.time() < deadline and public_url is None:
    time.sleep(2)
    with open(tunnel_log_path) as f:
        m = url_re.search(f.read())
    if m:
        public_url = m.group(0)

if not public_url:
    raise RuntimeError("cloudflared did not print a public URL. See logs/cloudflared.log")

banner = "\n".join([
    "",
    "╔" + "═" * 70 + "╗",
    "║" + "  GRANITE SPEECH DEMO IS LIVE".ljust(70) + "║",
    "╠" + "═" * 70 + "╣",
    "║" + f"  {public_url}".ljust(70) + "║",
    "║" + "".ljust(70) + "║",
    "║" + "  1. Open the URL above in Chrome / Edge / Firefox".ljust(70) + "║",
    "║" + "  2. Allow microphone access when prompted".ljust(70) + "║",
    "║" + "  3. Click the mic button and start talking".ljust(70) + "║",
    "╚" + "═" * 70 + "╝",
    "",
])
print(banner)

## 7 · If something goes wrong

Each background process writes to a file in `logs/`:

- `logs/vllm-speech.log` — Granite Speech STT server
- `logs/vllm-switch.log` — Granite Switch LLM server
- `logs/backend.log` — Pipecat backend (look here for TURN minting messages)
- `logs/frontend.log` — Next.js dev server
- `logs/cloudflared.log` — Cloudflare tunnel (the public URL is in here)

View one with `!tail -100 logs/vllm-speech.log` (or open the file from the Colab file browser).

**Common failures:**
- *GPU OOM:* switch the runtime to an A100. Both Granite models won't fit on smaller GPUs (T4, L4).
- *`HF_TOKEN` missing:* re-run Cell 3 after adding the secret. Without it, the backend falls back to STUN-only and audio likely won't connect through the cloudflared tunnel.
- *Stuck "waiting for vLLM":* model weights are downloading. The cell waits up to 20 min — let it run.
- *Re-running cells without cleaning up:* old processes still hold the ports. Run the kill-switch cell below, then re-run from the top.

## 8 · Caveats

- The `*.trycloudflare.com` URL is public for as long as this notebook runs. Anyone with the link can join the session.
- Colab kernels die after ~24h or when idle. Restart the notebook to get a fresh URL.
- One Colab session serves one user. Each reader runs their own copy of this notebook.

## 9 · Kill switch — clean up before re-running

Run this if you need to re-run any of the launch cells. It stops the tunnel, frontend, backend, and both vLLM processes.

**This cell is gated** so "Run all" won't tear down the stack you just brought up. To actually run it, uncomment the `RUN_KILL_SWITCH = True` line at the top of the cell, then run the cell.

In [ ]:
import subprocess

# Uncomment the next line to actually run the kill switch.
# This guard exists so "Run all" doesn't tear down the stack you just brought up.
# RUN_KILL_SWITCH = True

if not globals().get("RUN_KILL_SWITCH"):
    print("Kill switch is disabled. Uncomment `RUN_KILL_SWITCH = True` above and re-run this cell to stop all processes.")
else:
    # Stop tracked Popen handles from this kernel session.
    for name, p in [
        ("cloudflared", globals().get("tunnel_proc")),
        ("frontend", globals().get("frontend_proc")),
        ("backend", globals().get("backend_proc")),
        ("vllm-switch", globals().get("switch_proc")),
        ("vllm-speech", globals().get("speech_proc")),
    ]:
        if p is not None and p.poll() is None:
            p.terminate()
            try:
                p.wait(timeout=10)
            except Exception:
                p.kill()
            print(f"🛑 stopped {name}")
        else:
            print(f"   {name}: not tracked / already dead")

    # Also kill by name — catches processes whose Popen handles got lost across
    # cell re-runs or kernel restarts. Without this, GPU memory stays held by
    # zombie vllm processes and the next Cell 5 run fails with OOM at startup.
    # We run the frontend in prod mode (`npm run start` -> `next start`), so
    # match `next` rather than `next dev`.
    for pattern in ["vllm", "cloudflared tunnel", "granite_speech_demo.server", "next start", "node.*next"]:
        subprocess.run(f"pkill -9 -f '{pattern}' || true", shell=True)
        print(f"🧹 pkill -9 -f '{pattern}'")

    # Final safety net: free the ports the launch cells bind to. If a process
    # slipped past the name-based pkill above, this kills whatever is still
    # listening so re-running Cell 5 / Cell 6 doesn't fail with EADDRINUSE.
    #   3000 = Next.js frontend
    #   7860 = Pipecat backend
    #   8000 = Granite Switch vLLM
    #   8083 = Granite Speech vLLM
    for port in (3000, 7860, 8000, 8083):
        subprocess.run(f"fuser -k {port}/tcp 2>/dev/null || true", shell=True)
        print(f"🔓 freed port {port}")

    print("\nIf any vllm processes were running, GPU memory should now be freed.")
    print("Run `!nvidia-smi` to confirm before re-running Cell 5.")